In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import scanpy as sc

# Change path to wherever you have repo locally
sys.path.append('/oak/stanford/groups/engreitz/Users/ymo/Tools/cNMF_benchmarking/cNMF_benchmarking_pipeline')

from torch_cnmf import cNMF

from Inference.src import (
    run_cnmf_consensus, get_top_indices_fast, annotate_genes_to_excel,
    rename_and_move_files_NMF, rename_all_NMF, compile_results
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch

# Check if CUDA is available
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

# Get current GPU details
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB")
    print(f"Memory Reserved: {torch.cuda.memory_reserved(0)/1024**2:.2f} MB")

CUDA Available: False
CUDA Version: 12.4
Number of GPUs: 0


In [ ]:
# cNMF parameters
n_iter = 10
num_highvar_genes = 5451  
components = [30, 50, 60, 80, 100, 200, 250, 300]
sel_threshs = [0.4, 0.8, 2.0]
seed = 14
beta_loss = 'frobenius'
init = "random"
mode = "batch"
algo = 'halsvar'
tol = 1e-7
batch_max_epoch = 1000
minibatch_max_epoch = 200
minibatch_size = 100000
minibatch_max_iter = 1000
minibatch_usage_tol = 0.005
minibatch_spectra_tol = 0.005
use_gpu = True
batch_hals_max_iter = 1000
batch_hals_tol = 0.005

# annotation parameters
sk_cd_refit = True
gene_num = 300
species = "human"
gene_names_key = "symbol"  # column in adata.var with gene names; set to None to use var_names
 
# keys 
categorical_key="sex"
guide_assignment_key = "guide_assignment"
guide_names_key = "guide_names"
guide_targets_key = "guide_targets" 

# IO
counts_fn = "/oak/stanford/groups/engreitz/Users/ymo/cc-perturb-seq/Data/IGVF_D0_example.h5ad"
output_directory = "/oak/stanford/groups/engreitz/Users/ymo/cc-perturb-seq/Results"
run_name = "111025_D0_IGVF_10iter_torch_halsvar_batch_e7_v100s_test"

# resource files
nmf_seeds_path = None

In [ ]:
if nmf_seeds_path is not None:
    nmf_seeds = np.load(nmf_seeds_path)
else:
    nmf_seeds = None

In [ ]:
adata = sc.read(counts_fn)

In [ ]:
# remove non-coding genes (gene with only Ensembl ID), given gene symbol list
Ensembl_start = 'ENSG'
Gene_symbol_key = 'symbol'
mask = ~adata.var[Gene_symbol_key].str.startswith(Ensembl_start)
adata = adata[:, mask].copy()

filtered_path = f'{output_directory}/{run_name}/adata_without_noncoding.h5ad'
adata.write(filtered_path)
counts_fn = filtered_path

In [ ]:
cnmf_obj = cNMF(output_dir=output_directory, name=run_name)

In [ ]:
cnmf_obj.prepare(counts_fn=counts_fn, components=components, n_iter=n_iter, densify=False, tpm_fn=None, num_highvar_genes=num_highvar_genes, genes_file=None, beta_loss=beta_loss, 
                algo=algo, mode=mode, tol=tol, n_jobs=-1, init="random",
                seed=seed, use_gpu=use_gpu, 
                alpha_usage=0.0, alpha_spectra=0.0, 
                l1_ratio_usage=0.0, l1_ratio_spectra=0.0,
                minibatch_usage_tol=minibatch_usage_tol, minibatch_spectra_tol=minibatch_spectra_tol,
                fp_precision='float', 
                batch_max_epoch=batch_max_epoch, batch_hals_tol=batch_hals_tol, batch_hals_max_iter=batch_hals_max_iter,
                minibatch_max_epoch=minibatch_max_epoch, minibatch_size=minibatch_size, minibatch_max_iter=minibatch_max_iter,
                sk_cd_refit=sk_cd_refit, nmf_seeds=nmf_seeds)

In [ ]:
cnmf_obj.factorize(skip_completed_runs=True)

In [ ]:
cnmf_obj.combine()

In [ ]:
cnmf_obj.k_selection_plot()  

In [ ]:
# Consensus plots with all k to choose thresh
run_cnmf_consensus(cnmf_obj, 
                   components=components, 
                   density_thresholds=sel_threshs)

In [ ]:
# Save all cNMF scores in separate mudata objects
compile_results(output_directory, run_name, components=components, sel_threshs=sel_threshs,
     guide_names_key=guide_names_key, guide_targets_key=guide_targets_key, categorical_key=categorical_key, 
     guide_assignment_key=guide_assignment_key, gene_names_key=gene_names_key)

In [8]:
# annotation for all K
os.makedirs((f'{output_directory}/{run_name}/Annotation'), exist_ok=True)
for i in sel_threshs:
    for k in components:
        df = pd.read_csv('{output_directory}/{run_name}/{run_name}.gene_spectra_score.k_{k}.dt_{sel_thresh}.txt'.format(
                                                                                output_directory=output_directory,
                                                                                run_name = run_name,
                                                                                k=k,
                                                                                sel_thresh = str(i).replace('.','_')),
                                                                                sep='\t', index_col=0)   
        overlap = get_top_indices_fast(df, gene_num=300)
        annotate_genes_to_excel(overlap, species = 'human', output_file = f'{output_directory}/{run_name}/Annotation/{k}_{i}.xlsx')

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Annotating column 'Program_1': 300 genes...


23 input query terms found dup hits:	[('CALM2P2', 2), ('CHD2', 2), ('MORF4L1P1', 2), ('EIF3A', 2), ('UBC', 2), ('TCEA1P2', 2), ('GTF2IP4'
1 input query terms found no hit:	['GTF2IP1-1']
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Annotating column 'Program_2': 300 genes...


30 input query terms found dup hits:	[('PRH1', 2), ('LINC02315', 2), ('EGFEM1P', 2), ('DLG2', 2), ('PCDH11Y', 2), ('AUTS2', 2), ('NALF1-I
9 input query terms found no hit:	['POLR2J3-1', 'ENSG00000267327', 'ENSG00000273432', 'ENSG00000291215', 'ENSG00000237461', 'HERC2P3-1
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


...

34 input query terms found dup hits:	[('ANKRD20A3P', 2), ('ANKRD20A2P', 2), ('ENSG00000290971', 2), ('ANKRD20A7P', 2), ('GALNT13', 2), ('
6 input query terms found no hit:	['ENSG00000240240', 'ENSG00000289084', 'ENSG00000255028', 'ENSG00000286895', 'CD99P1-1', 'ENSG000002
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Annotating column 'Program_49': 300 genes...


41 input query terms found dup hits:	[('GOLGA6D', 2), ('GOLGA6FP', 2), ('SUDS3P1', 4), ('LINC00882', 2), ('XACT', 2), ('ATP8B5P', 2), ('S
6 input query terms found no hit:	['ENSG00000249335', 'ENSG00000283633', 'ENSG00000272369', 'ENSG00000229191', 'ENSG00000287523', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


Annotating column 'Program_50': 300 genes...


21 input query terms found dup hits:	[('ENSG00000275496', 2), ('LINC01937', 2), ('STARD7-AS1', 2), ('ZNF45-AS1', 2), ('LINC01002', 2), ('
7 input query terms found no hit:	['SDHAP4-1', 'ENSG00000286351', 'MGC32805', 'ENSG00000251652', 'ENSG00000291215', 'CA5BP1-1', 'ZNF87
